# Demonstration 1: Capturing an OpenMDAO Assembly, editing it, and executing the modified assembly

## Part 1: Capturing the OpenMDAO Assembly

This is the first of three Jupyter Notebooks that show an example of the workflow pictured below:

![Process showing separation of creation, editing, and use of MDAO models](workflow.drawio.svg)

In this notebook engineer 1 is creating an OpenMDAO assembly that covers multiple use cases. In this example we are using the `PrepGeom` group from Aviary. 

<div class="alert alert-block alert-info">
<b>Note:</b> Thanks to Ken Moore from NASA for providing the code to set this example up.
</div>

In [8]:
import numpy as np

import openmdao.api as om

from aviary.interface.methods_for_level2 import AviaryProblem
from aviary.variable_info.enums import Verbosity
from aviary.variable_info.variables import Aircraft, Dynamic, Mission, Settings

import standard_evaluator as se

In [9]:
phase_info = {
    "pre_mission": {"include_takeoff": True, "optimize_mass": True},
    "climb": {
        "subsystem_options": {"core_aerodynamics": {"method": "computed"}},
        "user_options": {
            'fix_initial': False,
            'input_initial': True,
            "optimize_mach": True,
            "optimize_altitude": True,
            "use_polynomial_control": False,
            "num_segments": 6,
            "order": 3,
            "solve_for_distance": False,
            "initial_mach": (0.3, "unitless"),
            "final_mach": (0.79, "unitless"),
            "mach_bounds": ((0.1, 0.8), "unitless"),
            "initial_altitude": (35., "ft"),
            "final_altitude": (35000.0, "ft"),
            "altitude_bounds": ((0.0, 35000.0), "ft"),
            "throttle_enforcement": "path_constraint",
            "constrain_final": False,
            "fix_duration": False,
            "initial_bounds": ((0.0, 2.0), "min"),
            "duration_bounds": ((5.0, 50.0), "min"),
            "no_descent": False,
            "add_initial_mass_constraint": False,
        },
        "initial_guesses": {"time": ([0, 40.0], "min")},
    },
    "cruise": {
        "subsystem_options": {"core_aerodynamics": {"method": "computed"}},
        "user_options": {
            "optimize_mach": True,
            "optimize_altitude": True,
            "polynomial_control_order": 1,
            "use_polynomial_control": True,
            "num_segments": 1,
            "order": 3,
            "solve_for_distance": False,
            "initial_mach": (0.79, "unitless"),
            "final_mach": (0.79, "unitless"),
            "mach_bounds": ((0.79, 0.79), "unitless"),
            "initial_altitude": (35000.0, "ft"),
            "final_altitude": (35000.0, "ft"),
            "altitude_bounds": ((35000.0, 35000.0), "ft"),
            "throttle_enforcement": "boundary_constraint",
            "fix_initial": False,
            "constrain_final": False,
            "fix_duration": False,
            "initial_bounds": ((64.0, 192.0), "min"),
            "duration_bounds": ((60.0, 720.0), "min"),
        },
        "initial_guesses": {"time": ([128, 113], "min")},
    },
    "descent": {
        "subsystem_options": {"core_aerodynamics": {"method": "computed"}},
        "user_options": {
            "optimize_mach": True,
            "optimize_altitude": True,
            "use_polynomial_control": False,
            "num_segments": 5,
            "order": 3,
            "solve_for_distance": False,
            "initial_mach": (0.79, "unitless"),
            "final_mach": (0.3, "unitless"),
            "mach_bounds": ((0.2, 0.8), "unitless"),
            "initial_altitude": (35000.0, "ft"),
            "final_altitude": (35.0, "ft"),
            "altitude_bounds": ((0.0, 35000.0), "ft"),
            "throttle_enforcement": "path_constraint",
            "fix_initial": False,
            "constrain_final": True,
            "fix_duration": False,
            "initial_bounds": ((120., 800.), "min"),
            "duration_bounds": ((5.0, 35.0), "min"),
            "no_climb": True,
        },
        "initial_guesses": {"time": ([241, 30], "min")},
    },
    "post_mission": {
        "include_landing": True,
        "constrain_range": True,
        "target_range": (3375.0, "nmi"),
    },
}


prob = AviaryProblem()

prob.load_inputs("models/test_aircraft/aircraft_for_bench_FwFm.csv", phase_info, verbosity=2)

# Because we are setting the wing mass (i.e., overriding the calculation of wing mass), the
# subsystems that produce it are safe to delete.
prob.aviary_inputs.set_val(Aircraft.Wing.MASS, 18000, units='lbm')

prob.check_and_preprocess_inputs()
prob.add_pre_mission_systems()
prob.add_phases()
prob.add_post_mission_systems()
prob.link_phases()

#prob.add_driver("SNOPT", max_iter=50)
#prob.driver.opt_settings["Major optimality tolerance"] = 1e-4
#prob.driver.opt_settings["Major feasibility tolerance"] = 1e-4

prob.add_design_variables()

prob.add_objective(objective_type='fuel_burned')

prob.setup()

prob.set_initial_guesses()

# Either run the optimization, or just run the model.
prob.run_model()
#prob.run_aviary_problem()

prob.model.pre_mission.core_subsystems.core_mass.wing_group.list_vars(units=True, print_arrays=True)

import standard_evaluator as se
info = se.get_interface(prob.model)

# For thie Demo, we can delete this group:
#    pre_mission.core_subsystems.core_mass.wing_group
# The group contains 6 components that contribute to the wing
# weight calculation, which we override by setting it above.

print('done')

c:\Users\tf715b\Desktop\standard-evaluator\venv\Lib\site-packages\aviary\utils\process_input_decks.py:175: UserWarning: Variable 'aircraft:wing:BENDING_MATERIAL_MASS_SCALER' is not in meta_data nor in 'guess_names'. It will be ignored.
  warnings.warn(
c:\Users\tf715b\Desktop\standard-evaluator\venv\Lib\site-packages\dymos\phase\phase.py:897: OMDeprecationWarning:None: The method `add_polynomial_control` is deprecated and will be removed in Dymos 2.1. Please use `add_control` with the appropriate options to define a polynomial control.



Initial Guesses
actual_takeoff_mass 175400
rotation_mass 173646.0
operating_empty_mass 0
fuel_burn_per_passenger_mile 0.1
cruise_mass_final 140320.0
flight_duration 24064.400920558826
time_to_climb 810.6105062524786
climb_range 58.14158131465364
reserves 3000.0
User has specified Design.NUM_* passenger values but CrewPyaload.NUM_* has been left blank or set to zero.
Assuming they are equal to maintain backwards compatibility with GASP and FLOPS output files.
If you intended to have no passengers on this flight, please set Aircraft.CrewPayload.TOTAL_PAYLOAD_MASS to zero in aviary_values.

The following variables have been overridden:
  'aircraft:design:touchdown_mass  152800  lbm
  'aircraft:engine:mass  [7400.]  lbm
  'aircraft:fins:mass  0  lbm
  'aircraft:fuel:auxiliary_fuel_capacity  0  lbm
  'aircraft:fuel:fuselage_fuel_capacity  0  lbm
  'aircraft:fuel:total_capacity  45694  lbm
  'aircraft:fuselage:planform_area  1578.24  ft**2
  'aircraft:fuselage:wetted_area  4158.62  ft**2
  

In [10]:
inputs_to_delete = prob.model.pre_mission.core_subsystems.core_mass.wing_group.list_vars(units=True, print_arrays=True)
print(list(inputs_to_delete))

57 Variables(s) in 'pre_mission.core_subsystems.core_mass.wing_group'

varname                                           val                    io      units     prom_name                                     
------------------------------------------------  ---------------------  ------  --------  ----------------------------------------------
engine_pod_mass
  aircraft:electrical:mass                        [2463.87138047]        input   lbm       aircraft:electrical:mass                      
  aircraft:fuel:fuel_system_mass                  [669.57723863]         input   lbm       aircraft:fuel:fuel_system_mass                
  aircraft:hydraulics:mass                        [1086.69550641]        input   lbm       aircraft:hydraulics:mass                      
  aircraft:instruments:mass                       [601.16492884]         input   lbm       aircraft:instruments:mass                     
  aircraft:nacelle:mass                           [1971.38199541]        input   lbm 

In [57]:
inputs_to_delete[2][1]['prom_name']

'aircraft:hydraulics:mass'

Now that we have the OpenMDAO assembly we can show the values of it's inputs. This will be useful to compare with in the other parts of the demo.

In [54]:
find_name = "aircraft:electrical:mass"
find_name = "aircraft:fuel:fuel_system_mass"

In [59]:
def find_me(name, info):
    for ele in info.inputs:
        if name == ele.name:
            print(f"{name}: Found inputs: {ele.name}, Component <{info.name}>")
    for ele in info.outputs:
        if name == ele.name:
            print(f"{name}: Found outputs: {ele.name}, Component <{info.name}>")
    if info.class_type == 'GroupInfo':
        for key, value in info.promotions.items():
            for local_ele in value:
                if name == local_ele[0]:
                    print(f"{name}: Found promotions: {key}, Component <{info.name}>, promotions: <{local_ele}>")
        for (source, target) in info.linkage:
            if source == name:
                print(f"{name}: Found linkage: {source}, Target {target}, Component <{info.name}>")
        for ele_name in info.component_order:
            find_me(name, info.components[ele_name])

In [58]:
all_names = []
for ele in inputs_to_delete:
    all_names.append(ele[1]['prom_name'])
print(all_names)
inputs_to_delete[2][1]['prom_name']

['aircraft:electrical:mass', 'aircraft:fuel:fuel_system_mass', 'aircraft:hydraulics:mass', 'aircraft:instruments:mass', 'aircraft:nacelle:mass', 'aircraft:propulsion:total_engine_controls_mass', 'aircraft:engine:mass', 'aircraft:propulsion:total_starter_mass', 'aircraft:engine:thrust_reversers_mass', 'aircraft:engine:scaled_sls_thrust', 'aircraft:propulsion:total_scaled_sls_thrust', 'aircraft:engine:pod_mass', 'aircraft:wing:load_path_sweep_dist', 'aircraft:wing:thickness_to_chord_dist', 'aircraft:wing:chord_per_semispan', 'mission:design:gross_mass', 'aircraft:engine:pod_mass', 'aircraft:wing:aspect_ratio', 'aircraft:wing:aspect_ratio_reference', 'aircraft:wing:strut_bracing_factor', 'aircraft:wing:aeroelastic_tailoring_factor', 'aircraft:engine:wing_locations', 'aircraft:wing:thickness_to_chord', 'aircraft:wing:thickness_to_chord_reference', 'aircraft:wing:bending_material_factor', 'aircraft:wing:eng_pod_inertia_factor', 'aircraft:wing:composite_fraction', 'aircraft:wing:area', 'airc

'aircraft:hydraulics:mass'

In [60]:
#find_name = "mission:takeoff:final_mass"
for find_name in all_names:
    find_me(find_name, info)

aircraft:electrical:mass: Found inputs: aircraft:electrical:mass, Component <>
aircraft:electrical:mass: Found outputs: aircraft:electrical:mass, Component <>
aircraft:electrical:mass: Found promotions: pre_mission, Component <>, promotions: <('aircraft:electrical:mass', 'aircraft:electrical:mass')>
aircraft:electrical:mass: Found promotions: pre_mission, Component <>, promotions: <('aircraft:electrical:mass', 'aircraft:electrical:mass')>
aircraft:electrical:mass: Found promotions: pre_mission, Component <>, promotions: <('aircraft:electrical:mass', 'aircraft:electrical:mass')>
aircraft:electrical:mass: Found inputs: aircraft:electrical:mass, Component <pre_mission>
aircraft:electrical:mass: Found outputs: aircraft:electrical:mass, Component <pre_mission>
aircraft:electrical:mass: Found promotions: core_subsystems, Component <pre_mission>, promotions: <('aircraft:electrical:mass', 'aircraft:electrical:mass')>
aircraft:electrical:mass: Found promotions: core_subsystems, Component <pre_m

We can also look at the N2 diagram for this model. If we carefully look at this diagram we can see that there are two main components that create outputs from this assembly:

- `characteristic_lengths`
- `total_wetted_area`

We can also see that two components are only using inputs to the group, and are not depending on the preliminary calculations done in other components in this assembly. The two components only depending on inputs to the group are:

- `nacelles`
- `canard`

In [ ]:
import os
# Check if running in VS Code
if 'VSCODE_PID' in os.environ:
    display_in_notebook = False
else:
    display_in_notebook = True
om.n2(prob, 'before_NASA_big.n2.html', display_in_notebook=display_in_notebook,  )

The final step in this part of the workflow we are demonstrating is the engineer storing both the assembly information and the current state of the assembly in a JSON file (`demo_group_nasa.json`) and a HDF5 file (`state.h5`). 

<div class="alert alert-block alert-info">
<b>Note:</b> We use the HDF5 format for storing the state since it is a binary file format which is compact and ensure no loss of accuracy. There are many different viewers and editors available to edit HDF5. A nice viewer is <a href="https://myhdf5.hdfgroup.org/">myHDF5</a>.
</div>

In [ ]:
#se.save_assembly(prob, assembly_name='demo_group_nasa_big.json', state_name='state_big.h5')

Indexing used: timeseries.dt_dstau, ('dt_dstau', None, True), <class 'str'>, <class 'NoneType'>, <class 'bool'>
Indexing used: rhs_all.aircraft:design:base_area, ('parameter_vals:aircraft:design:base_area', MultiIndexer: (array([0]),), True), <class 'str'>, <class 'openmdao.utils.indexer.MultiIndexer'>, <class 'bool'>
Indexing used: rhs_all.aircraft:design:lift_dependent_drag_coeff_factor, ('parameter_vals:aircraft:design:lift_dependent_drag_coeff_factor', MultiIndexer: (array([0]),), True), <class 'str'>, <class 'openmdao.utils.indexer.MultiIndexer'>, <class 'bool'>
Indexing used: rhs_all.aircraft:design:subsonic_drag_coeff_factor, ('parameter_vals:aircraft:design:subsonic_drag_coeff_factor', MultiIndexer: (array([0]),), True), <class 'str'>, <class 'openmdao.utils.indexer.MultiIndexer'>, <class 'bool'>
Indexing used: rhs_all.aircraft:design:supersonic_drag_coeff_factor, ('parameter_vals:aircraft:design:supersonic_drag_coeff_factor', MultiIndexer: (array([0]),), True), <class 'str'>, 